# mdSAM plotting (updated)

Updates:
- Total (left), Surface (right)
- Plot **log10-transformed** values (not log axis)
- Statistics remain on the same transformed scale
- Replicate trajectories are correctly grouped by `Element + Rep`
- Mean ± SEM
- Significant proteins in one figure, remaining proteins in a second
- Saves SVG figures


In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

file = "/Users/aelazar/notebooks_for_paper/response_reviewers/20270728_mdSAM_Constructs.xlsx"
metric="Geometric Mean"

palette={"WT":"#d62728","MUT":"gray"}

sns.set_theme(style="ticks",context="paper",font_scale=1.2)

mut_stats={
"A4":(1.710,0.001),"APP":(1.959,0.0001),"CD34":(-0.041,0.939),
"CX3CL1":(0.129,0.618),"ERVW-1":(-0.126,0.774),"EVI2A":(0.009,0.991),
"HTR7":(-0.197,0.338),"HTR7_SNAPtm":(1.040,0.0001),"KCNE4":(0.238,0.250),
"KCNE4_SNAPtm":(0.962,0.002),"MANS4":(0.514,0.159),
"SIGL6":(-0.157,0.641),"TIMD4":(-0.165,0.758)
}

surface=pd.read_excel(file,sheet_name="Surface")
total=pd.read_excel(file,sheet_name="Total")

surface=surface[surface["Element"].isin(["WT","MUT"])]
total=total[total["Element"].isin(["WT","MUT"])]

keys=["Construct","Element","DNA","Rep"]

surface=(surface.groupby(keys,as_index=False)[metric]
         .mean().rename(columns={metric:"Surface"}))
total=(total.groupby(keys,as_index=False)[metric]
       .mean().rename(columns={metric:"Total"}))

df=surface.merge(total,on=keys)

# log-transform BEFORE plotting/statistics
df["logSurface"]=np.log10(df["Surface"])
df["logTotal"]=np.log10(df["Total"])
df["DNAlog"]=np.log10(df["DNA"])
# unique trajectory per condition/replicate
df["Trajectory"]=df["Element"]+"_Rep"+df["Rep"].astype(str)
df["logTrafficking"] = df["logSurface"] - df["logTotal"]
def panel(ax,sub,y):
    sns.lineplot(
        data=sub,
        x="DNA",
        y=y,
        hue="Element",
        units="Trajectory",
        estimator=None,
        palette=palette,
        alpha=.30,
        lw=1.2,
        legend=False,
        ax=ax
    )

    sns.stripplot(
        data=sub,
        x="DNA",
        y=y,
        hue="Element",
        dodge=True,
        palette=palette,
        alpha=.45,
        size=6,
        linewidth=0,
        legend=False,
        ax=ax
    )

    sns.pointplot(
        data=sub,
        x="DNA",
        y=y,
        hue="Element",
        estimator="mean",
        errorbar="se",
        dodge=.4,
        palette=palette,
        markers="_",
        linestyles="",
        errwidth=2,
        capsize=.18,
        legend=False,
        ax=ax
    )

    for s in ax.spines.values():
        s.set_visible(True)
    ax.tick_params(direction="out")

def make_figure(proteins,title,outfile):
    fig,axes=plt.subplots(len(proteins),2,figsize=(10,3.4*len(proteins)),constrained_layout=True)
    if len(proteins)==1:
        axes=[axes]
    for i,p in enumerate(proteins):
        sub=df[df["Construct"]==p]

        ax=axes[i][0]
        panel(ax,sub,"logTotal")
        ax.set_ylabel("log10(Total)")
        ax.set_title(p,fontweight="bold")

        ax=axes[i][1]
        panel(ax,sub,"logSurface")
        ax.set_ylabel("log10(Surface)")
        b,pv=mut_stats[p]
        ax.text(.98,.98,f"β={b:.2f}\np={pv:.3g}",
                transform=ax.transAxes,
                ha="right",va="top",fontsize=9)

    fig.suptitle(title,fontweight="bold",fontsize=16)
    fig.savefig(outfile,bbox_inches="tight")
    plt.show()

#main=["APP","A4","HTR7_SNAPtm","KCNE4_SNAPtm"]
# supp=sorted([p for p in df["Construct"].unique() if p not in main])

# make_figure(main,"Proteins with significant WT/MUT effects","Main_Figure.svg")
# make_figure(supp,"Remaining proteins","Supplementary_Figure.svg")


In [ ]:
for pro in ['APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm', 'A4']:

    fig, axes = plt.subplots(1, 3, figsize=(6, 2.5), constrained_layout=True)

    sns.pointplot(
        x='DNAlog',
        y='logSurface',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[1]
    )

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[0]
    )
    sns.pointplot(
        x='DNAlog',
        y='logTrafficking',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[2]
    )

    # Get mutation effect from the mixed model
    stat = results[
        (results["Protein"] == pro) &
        (results["Term"] == "Element[T.MUT]")
    ].iloc[0]

    beta = stat["Estimate"]
    p = stat["P-value"]

    # Format p-value
    if p < 1e-3:
        exp = int(np.floor(np.log10(p)))
        mantissa = p / 10**exp
        p_text = rf"{mantissa:.1f} × 10$^{{{exp}}}$"
    else:
        p_text = f"{p:.3g}"

    fig.suptitle(
        rf"{pro}",
        fontsize=11
    )

    # Clean up axes
    axes[0].set_xlabel(r'log$_{10}$(DNA)')
    axes[1].set_xlabel(r'')
    axes[2].set_xlabel(r'')
    axes[0].set_ylabel(r'log$_{10}$(Total)')
    axes[1].set_ylabel(r'log$_{10}$(Surface)')
    axes[2].set_ylabel(r'log$_{10}$(Surface)-log$_{10}$(Total)')
    axes[0].legend_.remove()
    axes[1].legend_.remove()
    axes[2].legend_.remove()
    axes[0].locator_params(axis='y', nbins=3)
    axes[1].locator_params(axis='y', nbins=3)
    axes[2].locator_params(axis='y', nbins=3)   
    axes[1].text(
    0.05, 0.95,
    rf"$\beta$ = {beta:.2f}" "\n" rf"$P$ = {p_text}",
    transform=axes[1].transAxes,
    ha="left",
    va="top",
    fontsize=10
)
    plt.savefig(
        f'/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/{pro}.svg',
        bbox_inches='tight',
        transparent=True,
        dpi=900
    )

    plt.show()

In [ ]:
df

In [ ]:
for pro in list(set.difference(set(df.Construct.drop_duplicates().to_list()),set(['APP','KCNE4_SNAPtm','HTR7_SNAPtm','A4']))):

    fig, axes = plt.subplots(1, 3, figsize=(6, 2.5), constrained_layout=True)

    sns.pointplot(
        x='DNAlog',
        y='logSurface',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[1]
    )

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[0]
    )
    sns.pointplot(
        x='DNAlog',
        y='logTrafficking',
        data=df[df.Construct == pro],
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[2]
    )

    # Get mutation effect from the mixed model
    stat = results[
        (results["Protein"] == pro) &
        (results["Term"] == "Element[T.MUT]")
    ].iloc[0]

    beta = stat["Estimate"]
    p = stat["P-value"]

    # Format p-value
    if p < 1e-3:
        exp = int(np.floor(np.log10(p)))
        mantissa = p / 10**exp
        p_text = rf"{mantissa:.1f} × 10$^{{{exp}}}$"
    else:
        p_text = f"{p:.3g}"

    fig.suptitle(
        rf"{pro}",
        fontsize=11
    )

    # Clean up axes
    axes[0].set_xlabel(r'log$_{10}$(DNA)')
    axes[1].set_xlabel(r'')
    axes[2].set_xlabel(r'')
    axes[0].set_ylabel(r'log$_{10}$(Total)')
    axes[1].set_ylabel(r'log$_{10}$(Surface)')
    axes[2].set_ylabel(r'log$_{10}$(Surface)-log$_{10}$(Total)')
    axes[0].legend_.remove()
    axes[1].legend_.remove()
    axes[2].legend_.remove()
    axes[0].locator_params(axis='y', nbins=3)
    axes[1].locator_params(axis='y', nbins=3)
    axes[2].locator_params(axis='y', nbins=3)   
    axes[1].text(
    0.05, 0.95,
    rf"$\beta$ = {beta:.2f}" "\n" rf"$P$ = {p_text}",
    transform=axes[1].transAxes,
    ha="left",
    va="top",
    fontsize=10
)
    plt.savefig(
        f'/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/{pro}.svg',
        bbox_inches='tight',
        transparent=True,
        dpi=900
    )

    plt.show()

In [ ]:
results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract mutation effects
forest = results[results["Term"] == "Element[T.MUT]"].copy()

# 95% confidence intervals
forest["Lower"] = forest["Estimate"] - 1.96 * forest["StdErr"]
forest["Upper"] = forest["Estimate"] + 1.96 * forest["StdErr"]

# Sort by effect size (largest effect first)
forest = forest.sort_values("Estimate", ascending=False).reset_index(drop=True)

# Plot
fig, ax = plt.subplots(figsize=(3.5, 4.5))

y = np.arange(len(forest))

ax.errorbar(
    forest["Estimate"],
    y,
    xerr=[
        forest["Estimate"] - forest["Lower"],
        forest["Upper"] - forest["Estimate"]
    ],
    fmt="o",
    capsize=3,
    color="black"
)

ax.axvline(0, color="gray", linestyle="--", linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(forest["Protein"])
ax.invert_yaxis()  # Largest effect at the top

ax.set_xlabel(r"Mutation effect ($\beta$)")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/all_beta_CI.svg',
    dpi=900,
    transparent=True,
    bbox_inches='tight'
)

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(6, 5),
    sharex='col',
    sharey='col',
    constrained_layout=True
)

proteins = ['APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm']

for i, pro in enumerate(proteins):

    data = df[df.Construct == pro]

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 0]
    )

    sns.pointplot(
        x='DNAlog',
        y='logSurface',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 1]
    )

    sns.pointplot(
        x='DNAlog',
        y='logTrafficking',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 2]
    )

    # Remove legends and axis labels
    for ax in axes[i]:
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.locator_params(axis='y', nbins=3)

    # Statistics
    stat = results[
        (results["Protein"] == pro) &
        (results["Term"] == "Element[T.MUT]")
    ].iloc[0]

    beta = stat["Estimate"]
    p = stat["P-value"]

    if p < 1e-3:
        exp = int(np.floor(np.log10(p)))
        mantissa = p / 10**exp
        p_text = rf"{mantissa:.1f} × 10$^{{{exp}}}$"
    else:
        p_text = f"{p:.3g}"

    axes[i, 1].text(
        0.05, 0.95,
        rf"$\beta$ = {beta:.2f}" "\n" rf"$P$ = {p_text}",
        transform=axes[i, 1].transAxes,
        ha="left",
        va="top",
        fontsize=11
    )

    # Protein name
    axes[i, 0].text(
        0.02, 0.98,
        pro,
        transform=axes[i, 0].transAxes,
        ha='left',
        va='top',
        fontsize=10,
        fontweight='bold'
    )

# Column titles
axes[0, 0].set_title('Total')
axes[0, 1].set_title('Surface')
axes[0, 2].set_title('Trafficking')

# Show y tick labels ONLY on middle row
for r in [0, 2]:
    for c in range(3):
        axes[r, c].tick_params(labelleft=False)

# Show x tick labels ONLY on bottom row
for r in [0, 1]:
    for c in range(3):
        axes[r, c].tick_params(labelbottom=False)

# One y-axis label per column (middle row only)
axes[1, 0].set_ylabel(r'log$_{10}$(Total)')
axes[1, 1].set_ylabel(r'log$_{10}$(Surface)')
axes[1, 2].set_ylabel(r'log$_{10}$(Surface)-log$_{10}$(Total)')

# One x-axis label (middle bottom panel)
axes[2, 1].set_xlabel(r'log$_{10}$(DNA)')

plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/all_proteins.svg',
    dpi=900,
    transparent=True,
    bbox_inches='tight'
)

plt.show()

In [ ]:
proteins = sorted(
    list(
        set(df.Construct.unique()) -
        {'APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm'}
    )
)

n = len(proteins)

fig, axes = plt.subplots(
    n, 3,
    figsize=(6, 1.3 * n),
    sharex='col',
    sharey='col',
    constrained_layout=True
)

# Make axes always 2D
if n == 1:
    axes = axes[np.newaxis, :]

for i, pro in enumerate(proteins):

    data = df[df.Construct == pro]

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 0]
    )

    sns.pointplot(
        x='DNAlog',
        y='logSurface',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 1]
    )

    sns.pointplot(
        x='DNAlog',
        y='logTrafficking',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        ax=axes[i, 2]
    )

    # Remove legends and axis labels
    for ax in axes[i]:
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.locator_params(axis='y', nbins=3)

    # Statistics
    stat = results[
        (results["Protein"] == pro) &
        (results["Term"] == "Element[T.MUT]")
    ].iloc[0]

    beta = stat["Estimate"]
    p = stat["P-value"]

    if p < 1e-3:
        exp = int(np.floor(np.log10(p)))
        mantissa = p / 10**exp
        p_text = rf"{mantissa:.1f} × 10$^{{{exp}}}$"
    else:
        p_text = f"{p:.3g}"

    axes[i, 1].text(
        0.05, 0.95,
        rf"$\beta$ = {beta:.2f}" "\n" rf"$P$ = {p_text}",
        transform=axes[i, 1].transAxes,
        ha="left",
        va="top",
        fontsize=10
    )

    # Protein name
    axes[i, 0].text(
        0.02, 0.98,
        pro,
        transform=axes[i, 0].transAxes,
        ha='left',
        va='top',
        fontsize=10,
        fontweight='bold'
    )

# Column titles
axes[0, 0].set_title('Total')
axes[0, 1].set_title('Surface')
axes[0, 2].set_title('Trafficking')

# Show y tick labels only on the middle row
middle = n // 2
for r in range(n):
    if r != middle:
        for c in range(3):
            axes[r, c].tick_params(labelleft=False)

# Show x tick labels only on the bottom row
for r in range(n - 1):
    for c in range(3):
        axes[r, c].tick_params(labelbottom=False)

# One y-axis label per column
axes[middle, 0].set_ylabel(r'log$_{10}$(Total)')
axes[middle, 1].set_ylabel(r'log$_{10}$(Surface)')
axes[middle, 2].set_ylabel(r'log$_{10}$(Surface)-log$_{10}$(Total)')

# One x-axis label
axes[-1, 1].set_xlabel(r'log$_{10}$(DNA)')

plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/other_proteins.svg',
    dpi=900,
    transparent=False,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# import numpy as np
# import pandas as pd
# import statsmodels.formula.api as smf

# # =====================================================
# # Settings
# # =====================================================
# file = "20270728_mdSAM_Constructs.xlsx"
# metric = "Geometric Mean"

# protein_col = "Construct"
# condition_col = "Element"

# # =====================================================
# # Read Surface data
# # =====================================================
# df = pd.read_excel(file, sheet_name="Surface")

# # WT/MUT only
# df = df[df[condition_col].isin(["WT", "MUT"])].copy()

# # =====================================================
# # Average technical replicates
# # =====================================================
# df = (
#     df.groupby(
#         [protein_col, condition_col, "DNA", "Rep"],
#         as_index=False
#     )[metric]
#     .mean()
# )

# # =====================================================
# # Transform variables
# # =====================================================
# # Use log10 so the model matches the figures
# df["logSurface"] = np.log10(df[metric])

# # DNA as a continuous predictor
# df["logDNA"] = np.log10(df["DNA"])

# df[condition_col] = pd.Categorical(
#     df[condition_col],
#     categories=["WT", "MUT"]
# )

# results = []

# # =====================================================
# # Mixed model for every protein
# # =====================================================
# for protein in sorted(df[protein_col].unique()):

#     sub = df[df[protein_col] == protein].copy()

#     if sub["Rep"].nunique() < 2:
#         continue

#     try:

#         model = smf.mixedlm(
#             "logSurface ~ Element * logDNA",
#             data=sub,
#             groups=sub["Rep"]
#         )

#         fit = model.fit(
#             method="lbfgs",
#             reml=True
#         )

#         print("\n")
#         print("=" * 70)
#         print(protein)
#         print("=" * 70)
#         print(fit.summary())

#         for term in fit.params.index:

#             results.append({
#                 "Protein": protein,
#                 "Term": term,
#                 "Estimate": fit.params[term],
#                 "StdErr": fit.bse[term],
#                 "P-value": fit.pvalues[term]
#             })

#     except Exception as e:

#         print(f"{protein}: model failed ({e})")

# # =====================================================
# # Summary table
# # =====================================================
# results = pd.DataFrame(results)

# print("\nSummary\n")
# print(results)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =====================================================
# Settings
# =====================================================
file = "20270728_mdSAM_Constructs.xlsx"
metric = "Geometric Mean"

protein_col = "Construct"
condition_col = "Element"

# =====================================================
# Read Surface and Total data
# =====================================================
surface = pd.read_excel(file, sheet_name="Surface")
total = pd.read_excel(file, sheet_name="Total")

# Keep only WT and MUT
surface = surface[surface[condition_col].isin(["WT", "MUT"])].copy()
total = total[total[condition_col].isin(["WT", "MUT"])].copy()

# =====================================================
# Average technical replicates
# =====================================================
surface = (
    surface.groupby(
        [protein_col, condition_col, "DNA", "Rep"],
        as_index=False
    )[metric]
    .mean()
    .rename(columns={metric: "Surface"})
)

total = (
    total.groupby(
        [protein_col, condition_col, "DNA", "Rep"],
        as_index=False
    )[metric]
    .mean()
    .rename(columns={metric: "Total"})
)

# =====================================================
# Merge Surface and Total
# =====================================================
df = pd.merge(
    surface,
    total,
    on=[protein_col, condition_col, "DNA", "Rep"],
    how="inner"
)

# =====================================================
# Log transforms
# =====================================================
df["logSurface"] = np.log10(df["Surface"])
df["logTotal"] = np.log10(df["Total"])
df["logDNA"] = np.log10(df["DNA"])

df[condition_col] = pd.Categorical(
    df[condition_col],
    categories=["WT", "MUT"]
)

results = []

# =====================================================
# Mixed model for each construct
# =====================================================
for protein in sorted(df[protein_col].unique()):

    sub = df[df[protein_col] == protein].copy()

    if sub["Rep"].nunique() < 2:
        continue

    try:

        model = smf.mixedlm(
            "logSurface ~ Element * logDNA + logTotal",
            data=sub,
            groups=sub["Rep"]
        )

        fit = model.fit(
            method="lbfgs",
            reml=True
        )

        print("\n")
        print("=" * 70)
        print(protein)
        print("=" * 70)
        print(fit.summary())

        for term in fit.params.index:

            results.append({
                "Protein": protein,
                "Term": term,
                "Estimate": fit.params[term],
                "StdErr": fit.bse[term],
                "P-value": fit.pvalues[term]
            })

    except Exception as e:

        print(f"{protein}: model failed ({e})")

# =====================================================
# Summary table
# =====================================================
results = pd.DataFrame(results)

print("\nSummary\n")
print(results)

# Optional: save results
#results.to_csv("MixedModel_Surface_AdjustedForTotal.csv", index=False)

In [ ]:
results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract mutation effects
forest = results[results["Term"] == "Element[T.MUT]"].copy()

# 95% confidence intervals
forest["Lower"] = forest["Estimate"] - 1.96 * forest["StdErr"]
forest["Upper"] = forest["Estimate"] + 1.96 * forest["StdErr"]

# Sort by effect size (largest effect first)
forest = forest.sort_values("Estimate", ascending=False).reset_index(drop=True)

# Plot
fig, ax = plt.subplots(figsize=(3.5, 4.5))

y = np.arange(len(forest))

ax.errorbar(
    forest["Estimate"],
    y,
    xerr=[
        forest["Estimate"] - forest["Lower"],
        forest["Upper"] - forest["Estimate"]
    ],
    fmt="o",
    capsize=3,
    color="black"
)

ax.axvline(0, color="gray", linestyle="--", linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(forest["Protein"])
ax.invert_yaxis()  # Largest effect at the top

ax.set_xlabel(r"Mutation effect ($\beta$)")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/all_beta_CI.svg',
    dpi=900,
    transparent=True,
    bbox_inches='tight'
)

In [ ]:
import math

proteins = sorted(
    list(
        set(df.Construct.unique()) -
        {'APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm'}
    )
)

n = len(proteins)
nrows = 2
ncols = math.ceil(n / nrows)
xticks = sorted(df["DNAlog"].unique())
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(2.3 * ncols, 2.4 * nrows),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

axes = np.array(axes).flatten()

for i, pro in enumerate(proteins):

    data = df[df.Construct == pro]

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,dodge=False,
        ax=axes[i]
    )

    if axes[i].legend_ is not None:
        axes[i].legend_.remove()

    axes[i].set_title(pro, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')
# Get the actual DNA concentrations used

    
    for ax in axes[:n]:
        ax.set_xticks(xticks)
        ax.locator_params(axis='y', nbins=3)
        ax.set_xticklabels([f"{x:.2f}" for x in xticks])  # optional formatting    # axes[i].locator_params(axis='y', nbins=3)

# Remove any unused axes
for j in range(n, len(axes)):
    fig.delaxes(axes[j])

# Axis labels
# for ax in axes[ncols:]:
#     ax.set_xlabel(r'log$_{10}$(DNA)')

# for ax in axes[::ncols]:
#     ax.set_ylabel(r'log$_{10}$(Total)')

plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/total_expression_only.svg',
    dpi=900,
    bbox_inches='tight'
)

plt.show()

In [ ]:
total_results = []

for protein in sorted(df[protein_col].unique()):

    sub = df[df[protein_col] == protein].copy()

    if sub["Rep"].nunique() < 2:
        continue

    try:

        model = smf.mixedlm(
            "logTotal ~ Element * logDNA",
            data=sub,
            groups=sub["Rep"]
        )

        fit = model.fit(method="lbfgs", reml=True)

        if "Element[T.MUT]" in fit.params.index:

            total_results.append({
                "Protein": protein,
                "Estimate": fit.params["Element[T.MUT]"],
                "StdErr": fit.bse["Element[T.MUT]"],
                "P-value": fit.pvalues["Element[T.MUT]"]
            })

    except Exception as e:
        print(f"{protein}: model failed ({e})")

total_results = pd.DataFrame(total_results)

print(total_results)

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(6, 5),
    sharex='col',
    sharey='col',
    constrained_layout=True
)

proteins = ['APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm']

# Use the actual DNA concentrations as x ticks
xticks = sorted(df["DNAlog"].unique())

for i, pro in enumerate(proteins):

    data = df[df.Construct == pro]

    sns.pointplot(
        x='DNAlog',
        y='logTotal',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        dodge=False,
        ax=axes[i, 0]
    )

    sns.pointplot(
        x='DNAlog',
        y='logSurface',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        dodge=False,
        ax=axes[i, 1]
    )

    sns.pointplot(
        x='DNAlog',
        y='logTrafficking',
        data=data,
        hue='Element',
        errorbar='se',
        palette=['#AF3437', '#BDBEC0'],
        hue_order=['WT', 'MUT'],
        native_scale=True,
        dodge=False,
        ax=axes[i, 2]
    )

    # Remove legends and format axes
    for ax in axes[i]:
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.locator_params(axis='y', nbins=3)
        ax.set_xticks(xticks)
        ax.set_xticklabels([f"{x:.2f}" for x in xticks])

    # Statistics
    stat = results[
        (results["Protein"] == pro) &
        (results["Term"] == "Element[T.MUT]")
    ].iloc[0]

    beta = stat["Estimate"]
    p = stat["P-value"]

    exp = int(np.floor(np.log10(p)))
    mantissa = p / 10**exp

    axes[i, 1].text(
        0.05, 0.95,
        rf"$\beta$ = {beta:.2f}" "\n"
        rf"$P = {mantissa:.2f}\times10^{{{exp}}}$",
        transform=axes[i, 1].transAxes,
        ha="left",
        va="top",
        fontsize=9
    )

    # Protein name
    axes[i, 0].text(
        0.02, 0.98,
        pro,
        transform=axes[i, 0].transAxes,
        ha='left',
        va='top',
        fontsize=10,
        fontweight='bold'
    )

# Column titles
axes[0, 0].set_title('Total')
axes[0, 1].set_title('Surface')
axes[0, 2].set_title('Trafficking')

# Show y tick labels ONLY on middle row
for r in [0, 2]:
    for c in range(3):
        axes[r, c].tick_params(labelleft=False)

# Show x tick labels ONLY on bottom row
for r in [0, 1]:
    for c in range(3):
        axes[r, c].tick_params(labelbottom=False)

# Axis labels
axes[1, 0].set_ylabel(r'log$_{10}$(Total)')
axes[1, 1].set_ylabel(r'log$_{10}$(Surface)')
axes[1, 2].set_ylabel(r'log$_{10}$(Surface)-log$_{10}$(Total)')
axes[2, 1].set_xlabel(r'log$_{10}$(DNA)')

plt.savefig(
    '/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/all_proteins.svg',
    dpi=900,
    transparent=True,
    bbox_inches='tight'
)

plt.show()

In [ ]:
df

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

proteins = sorted(
    list(
        set(df.Construct.unique()) -
        {'APP', 'KCNE4_SNAPtm', 'HTR7_SNAPtm'}
    )
)

n = len(proteins)
nrows = 2
ncols = math.ceil(n / nrows)

xticks = sorted(df["DNAlog"].unique())

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(2.4 * ncols, 2.8 * nrows),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

axes = np.array(axes).flatten()

colors = {"WT": "#AF3437", "MUT": "#BDBEC0"}
bar_width = 0.35

for i, pro in enumerate(proteins):

    ax = axes[i]
    data = df[df.Construct == pro]

    x = np.arange(len(xticks))

    wt_means, wt_sem = [], []
    mut_means, mut_sem = [], []

    for dna in xticks:

        wt = data[(data.DNAlog == dna) & (data.Element == "WT")]["logTotal"]
        mut = data[(data.DNAlog == dna) & (data.Element == "MUT")]["logTotal"]

        wt_means.append(wt.mean())
        wt_sem.append(wt.sem())

        mut_means.append(mut.mean())
        mut_sem.append(mut.sem())

    ax.bar(
        x - bar_width/2,
        wt_means,
        width=bar_width,
        yerr=wt_sem,
        color=colors["WT"],
        capsize=2,
        label="WT"
    )

    ax.bar(
        x + bar_width/2,
        mut_means,
        width=bar_width,
        yerr=mut_sem,
        color=colors["MUT"],
        capsize=2,
        label="Proline mut"
    )

    ymax = max(max(wt_means), max(mut_means))

    for j, dna in enumerate(xticks):

        wt = data[(data.DNAlog == dna) & (data.Element == "WT")]["logTotal"]
        mut = data[(data.DNAlog == dna) & (data.Element == "MUT")]["logTotal"]

        _, p = ttest_ind(wt, mut)  # two-sided Student's t-test

        y = max(
            wt_means[j] + wt_sem[j],
            mut_means[j] + mut_sem[j]
        ) + 0.08 * ymax

        # significance bracket
        ax.plot(
            [j - bar_width/2, j - bar_width/2,
             j + bar_width/2, j + bar_width/2],
            [y, y + 0.02*ymax,
             y + 0.02*ymax, y],
            color="black",
            lw=1
        )

        # p-value annotation
        if p < 0.001:
            p_text = "P < 0.001"
        else:
            p_text = f"P = {p:.2f}"

        ax.text(
            j,
            y + 0.025*ymax,
            p_text,
            ha="center",
            va="bottom",
            fontsize=8
        )

    ax.set_title(pro, fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.2f}" for v in xticks])
    ax.locator_params(axis="y", nbins=3)
    ax.set_xlabel("")
    ax.set_ylabel("")

# Remove unused panels
for j in range(n, len(axes)):
    fig.delaxes(axes[j])

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=False
)

fig.supxlabel("log$_{10}$(DNA)")
fig.supylabel("log$_{10}$(Total expression)")

plt.savefig(
    "/Users/aelazar/notebooks_for_paper/response_reviewers/total_vs_surface/total_expression_barplot.svg",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
groups = ["Red", "Black"]

# Emax
emax_mean = [99.8, 75.3]
emax_ci = [4.6, 4.8]
p_emax = 0.0003

# EC50 (nM)
ec50_mean = [2.69, 2.79]
ec50_ci = [0.94, 1.26]
p_ec50 = 0.95


def format_pvalue(p):
    """Return p-value formatted as scientific notation if small."""
    if p < 0.01:
        mantissa, exp = f"{p:.2e}".split("e")
        mantissa = float(mantissa)
        exp = int(exp)
        return rf"$P = {mantissa:.2f}\times10^{{{exp}}}$"
    else:
        return rf"$P = {p:.2f}$"


# Create figure
fig, axes = plt.subplots(1, 2, figsize=(4, 2.5))

x = np.arange(len(groups))

# -----------------------
# Emax plot
# -----------------------
axes[0].bar(x, emax_mean, yerr=emax_ci, capsize=6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(groups)
axes[0].set_ylabel("Emax")
axes[0].set_title("Emax (95% CI)")
axes[0].set_ylim(0, 120)
axes[0].locator_params(axis='y',nbins=4)
axes[0].text(
    0.5,
    110,
    format_pvalue(p_emax),
    ha="center",
    fontsize=12,
)

# -----------------------
# EC50 plot
# -----------------------
axes[1].bar(x, ec50_mean, yerr=ec50_ci, capsize=6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(groups)
axes[1].set_ylabel("EC50 (nM)")
axes[1].set_title("EC50 (95% CI)")
axes[1].set_ylim(0, 5)
axes[1].locator_params(axis='y',nbins=3)
axes[1].text(
    0.5,
    4.6,
    format_pvalue(p_ec50),
    ha="center",
    fontsize=12,
)

plt.tight_layout()
plt.savefig(
    "/Users/aelazar/notebooks_for_paper/response_reviewers/HTR7_stats_emax_ec50.svg",
    dpi=900,
    bbox_inches="tight"
)

In [ ]:
plt.)()